# CLEIDS-Edge — Notebook 00: Setup and Data Acquisition

Repo/Drive bootstrap + acquisition of the four benchmark datasets (NSL-KDD, CICIDS2017, IoT-23, UNSW-NB15, TON_IoT) used throughout the CLEIDS-Edge thesis pipeline.

See `CLEIDS_PROJECT_BRIEF.md` at the repo root for full system context. This notebook must run start to finish before Notebook 01 begins — no fabricated numbers, no substituted data; anything that can't be acquired automatically is reported as a manual action, never skipped silently.

**Required Colab secrets** (key icon, left sidebar): `GITHUB_TOKEN` (repo write access), `KAGGLE_USERNAME` + `KAGGLE_KEY` (from kaggle.com → Account → Create New API Token).

## 1. Repo setup (clone/pull + auth)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found. Add it to Colab secrets (key icon, left sidebar) as a "
        "GitHub personal access token with repo write access."
    )

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", AUTH_REMOTE, REPO_DIR], check=True)

# Keep the authenticated remote set for later pushes this session
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 2. Google Drive mount (persistence for models/results only)

Raw datasets are **not** cached to Drive -- per session preference, they live only on the Colab VM's local disk (`/content/...`) and are re-downloaded each fresh runtime. Drive is used only for `models/` and `results/` in later notebooks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_DATA_RAW = os.path.join(DRIVE_ROOT, "data", "raw")
DRIVE_MODELS = os.path.join(DRIVE_ROOT, "models")
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")

for d in (DRIVE_DATA_RAW, DRIVE_MODELS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

print("Drive persistence root ready at:", DRIVE_ROOT)

## 3. Local repo folder structure + .gitignore

In [ ]:
LOCAL_DIRS = ["notebooks", "data/raw", "data/processed", "models", "results", "figures"]
for d in LOCAL_DIRS:
    os.makedirs(d, exist_ok=True)

# data/ is bulk-ignored (raw datasets restored from Drive each session), but the manifest
# this notebook produces must still be tracked in git per the project brief.
GITIGNORE_RULES = [
    "data/*",
    "!data/dataset_manifest.json",
    "*.h5",
    "*.keras",
    "*.tflite",
    "*.ckpt",
    "*.weights.h5",
    "__pycache__/",
    ".ipynb_checkpoints/",
]

GITIGNORE_PATH = ".gitignore"
if not os.path.exists(GITIGNORE_PATH):
    with open(GITIGNORE_PATH, "w") as f:
        f.write("\n".join(GITIGNORE_RULES) + "\n")
    print("Created .gitignore")
else:
    existing = open(GITIGNORE_PATH).read()
    missing = [r for r in GITIGNORE_RULES if r not in existing]
    if missing:
        with open(GITIGNORE_PATH, "a") as f:
            f.write("\n" + "\n".join(missing) + "\n")
        print("Appended missing rules to .gitignore:", missing)
    else:
        print(".gitignore already covers required rules")

print("Local folder structure ready:", LOCAL_DIRS)

## 4. Environment check (CPU-safe — this notebook does not require a GPU)

In [ ]:
import platform
import tensorflow as tf

print("Python version:", platform.python_version())
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU visible: {gpus}")
else:
    print("No GPU visible -- expected and fine for this setup notebook (CPU-only).")

## 5. Package installation (via `uv` for fast, only-if-missing installs)

In [ ]:
import importlib
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

REQUIRED = {
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "imblearn": "imbalanced-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "kaggle": "kaggle",
}

missing = []
for import_name, package_name in REQUIRED.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print("Installing missing packages with uv:", missing)
    subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "--system", "-q", *missing],
        check=True,
    )
else:
    print("All required packages already present.")

## 6. Kaggle API auth

Two of the four datasets no longer have a scriptable official download (see §7 markdown notes below), so they are sourced via the Kaggle API as clearly-labeled mirrors.

In [ ]:
import json
import stat

try:
    from google.colab import userdata
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
except Exception:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME")
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY")

KAGGLE_READY = bool(KAGGLE_USERNAME and KAGGLE_KEY)

if KAGGLE_READY:
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
    with open(kaggle_json_path, "w") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod(kaggle_json_path, stat.S_IRUSR | stat.S_IWUSR)
    print("Kaggle credentials configured.")
else:
    print(
        "KAGGLE_USERNAME / KAGGLE_KEY not found in Colab secrets. NSL-KDD, UNSW-NB15, and "
        "TON_IoT are sourced via Kaggle and will be skipped with manual-download "
        "instructions until these secrets are added."
    )

## 7. Dataset acquisition

### Helpers

In [ ]:
import shutil

dataset_status = {}

def already_have(local_dir):
    return os.path.isdir(local_dir) and any(os.scandir(local_dir))

def restore_from_drive(name, local_dir):
    """Return True if data is already local or was restored from the Drive cache."""
    if already_have(local_dir):
        return True
    drive_dir = os.path.join(DRIVE_DATA_RAW, name)
    if os.path.isdir(drive_dir) and any(os.scandir(drive_dir)):
        print(f"[{name}] Restoring from Drive cache...")
        shutil.copytree(drive_dir, local_dir, dirs_exist_ok=True)
        return True
    return False

def record_ready(name, source_type, source, target_dir):
    dataset_status[name] = {
        "status": "ready", "source_type": source_type, "source": source, "path": target_dir,
    }
    print(f"[{name}] Ready at {target_dir}")

def record_manual(name, reason, target_dir, instructions):
    os.makedirs(target_dir, exist_ok=True)
    dataset_status[name] = {
        "status": "manual_required", "source_type": "manual", "path": target_dir,
        "note": reason, "instructions": instructions,
    }
    print(f"[{name}] MANUAL ACTION REQUIRED: {reason}")
    print(f"  -> {instructions}")

def record_failed(name, source_type, source, target_dir, error):
    dataset_status[name] = {
        "status": "failed", "source_type": source_type, "source": source,
        "path": target_dir, "error": str(error),
    }
    print(f"[{name}] Download failed: {error}")

### 7a. NSL-KDD

**Official source is dead**: `unb.ca/cic/datasets/nsl.html` currently returns "we apologize, this dataset is no longer available," and NSL-KDD is absent from the current CIC dataset index. Sourced via Kaggle mirror `hassan06/nslkdd` instead (logged in the manifest as `kaggle-mirror`, not `official-direct`, so this is traceable in the thesis methodology).

In [ ]:
name = "nsl-kdd"
target_dir = "data/raw/nsl-kdd"
KAGGLE_SLUG_NSLKDD = "hassan06/nslkdd"

if restore_from_drive(name, target_dir):
    record_ready(name, "kaggle-mirror", KAGGLE_SLUG_NSLKDD, target_dir)
elif not KAGGLE_READY:
    record_manual(
        name,
        "Official CIC page no longer hosts NSL-KDD, and Kaggle credentials are not configured.",
        target_dir,
        f"Add KAGGLE_USERNAME/KAGGLE_KEY to Colab secrets, or manually download '{KAGGLE_SLUG_NSLKDD}' from kaggle.com and place the files in {target_dir}/.",
    )
else:
    try:
        os.makedirs(target_dir, exist_ok=True)
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", KAGGLE_SLUG_NSLKDD,
             "-p", target_dir, "--unzip"],
            check=True,
        )
        record_ready(name, "kaggle-mirror", KAGGLE_SLUG_NSLKDD, target_dir)
    except Exception as e:
        record_failed(name, "kaggle-mirror", KAGGLE_SLUG_NSLKDD, target_dir, e)

### 7b. CICIDS2017

Official direct download from the Canadian Institute for Cybersecurity, no login required. **Note:** CIC periodically restructures file paths on `cicresearch.ca` — if the URL below 404s, verify the current path at [the official IDS-2017 page](https://www.unb.ca/cic/datasets/ids-2017.html) and update `CICIDS_URL`; the cell falls back to manual instructions rather than guessing.

In [ ]:
import urllib.request
import zipfile

name = "cicids2017"
target_dir = "data/raw/cicids2017"
CICIDS_URL = "http://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/MachineLearningCSV.zip"

if restore_from_drive(name, target_dir):
    record_ready(name, "official-direct", CICIDS_URL, target_dir)
else:
    try:
        os.makedirs(target_dir, exist_ok=True)
        zip_path = os.path.join(target_dir, "MachineLearningCSV.zip")
        print("Downloading CICIDS2017 from the official CIC source (large file)...")
        urllib.request.urlretrieve(CICIDS_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target_dir)
        os.remove(zip_path)
        record_ready(name, "official-direct", CICIDS_URL, target_dir)
    except Exception as e:
        record_manual(
            name,
            f"Automatic download from the official CIC source failed: {e}",
            target_dir,
            f"Manually download MachineLearningCSV.zip from https://www.unb.ca/cic/datasets/ids-2017.html and extract the CSVs into {target_dir}/.",
        )

### 7c. IoT-23

Official direct download from Stratosphere Laboratory's own mirror, no login required (small variant, ~8.7GB).

In [ ]:
import tarfile

name = "iot-23"
target_dir = "data/raw/iot-23"
IOT23_URL = "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz"

if restore_from_drive(name, target_dir):
    record_ready(name, "official-direct", IOT23_URL, target_dir)
else:
    try:
        os.makedirs(target_dir, exist_ok=True)
        tar_path = os.path.join(target_dir, "iot_23_datasets_small.tar.gz")
        print("Downloading IoT-23 (small, ~8.7GB) from the official Stratosphere mirror...")
        urllib.request.urlretrieve(IOT23_URL, tar_path)
        with tarfile.open(tar_path) as tar:
            tar.extractall(target_dir)
        os.remove(tar_path)
        record_ready(name, "official-direct", IOT23_URL, target_dir)
    except Exception as e:
        record_manual(
            name,
            f"Automatic download from the official Stratosphere mirror failed: {e}",
            target_dir,
            f"Manually download from https://www.stratosphereips.org/datasets-iot23 "
            f"and extract into {target_dir}/.",
        )

### 7d. UNSW-NB15

**Official source is not scriptable**: `research.unsw.edu.au/projects/unsw-nb15-dataset` links to a SharePoint folder requiring interactive Microsoft login. Sourced via Kaggle mirror `mrwellsdavid/unsw-nb15` instead (logged as `kaggle-mirror` in the manifest).

In [ ]:
name = "unsw-nb15"
target_dir = "data/raw/unsw-nb15"
KAGGLE_SLUG_UNSW = "mrwellsdavid/unsw-nb15"

if restore_from_drive(name, target_dir):
    record_ready(name, "kaggle-mirror", KAGGLE_SLUG_UNSW, target_dir)
elif not KAGGLE_READY:
    record_manual(
        name,
        "Official UNSW page links to a SharePoint folder needing interactive login, and "
        "Kaggle credentials are not configured.",
        target_dir,
        f"Add KAGGLE_USERNAME/KAGGLE_KEY to Colab secrets, or manually download from "
        f"https://research.unsw.edu.au/projects/unsw-nb15-dataset and place the CSVs in "
        f"{target_dir}/.",
    )
else:
    try:
        os.makedirs(target_dir, exist_ok=True)
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", KAGGLE_SLUG_UNSW,
             "-p", target_dir, "--unzip"],
            check=True,
        )
        record_ready(name, "kaggle-mirror", KAGGLE_SLUG_UNSW, target_dir)
    except Exception as e:
        record_failed(name, "kaggle-mirror", KAGGLE_SLUG_UNSW, target_dir, e)

### 7e. TON_IoT

Same access constraint as UNSW-NB15 (SharePoint, interactive login). Sourced via Kaggle mirror `alaaelmor/ton-iot-train-test-network` (mirrors the official `Train_Test_Network.csv` split).

In [ ]:
name = "ton-iot"
target_dir = "data/raw/ton-iot"
KAGGLE_SLUG_TONIOT = "alaaelmor/ton-iot-train-test-network"

if restore_from_drive(name, target_dir):
    record_ready(name, "kaggle-mirror", KAGGLE_SLUG_TONIOT, target_dir)
elif not KAGGLE_READY:
    record_manual(
        name,
        "Official UNSW page links to a SharePoint folder needing interactive login, and "
        "Kaggle credentials are not configured.",
        target_dir,
        f"Add KAGGLE_USERNAME/KAGGLE_KEY to Colab secrets, or manually download from "
        f"https://research.unsw.edu.au/projects/toniot-datasets and place the CSVs in "
        f"{target_dir}/.",
    )
else:
    try:
        os.makedirs(target_dir, exist_ok=True)
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", KAGGLE_SLUG_TONIOT,
             "-p", target_dir, "--unzip"],
            check=True,
        )
        record_ready(name, "kaggle-mirror", KAGGLE_SLUG_TONIOT, target_dir)
    except Exception as e:
        record_failed(name, "kaggle-mirror", KAGGLE_SLUG_TONIOT, target_dir, e)

## 8. Sanity checks

Report-only: row/column counts, label distribution, missing values, duplicate rows. Fixing (imputation, dedup, rebalancing) happens in Notebook 01, not here.

In [ ]:
import pandas as pd

LABEL_COLUMN_CANDIDATES = ["label", "Label", " Label", "class", "attack_cat", "type"]
DUP_CHECK_ROW_LIMIT = 2_000_000  # exact duplicate check skipped past this size per file

# NSL-KDD's KDDTrain+.txt/KDDTest+.txt are headerless -- pd.read_csv would silently
# treat the first data row as a header and misdetect the label column entirely.
# Column names are the well-known 41-feature + label + difficulty NSL-KDD schema.
NSLKDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label",
    "difficulty",
]
# Only the canonical full train/test files -- KDDTrain+_20Percent.txt and
# KDDTest-21.txt are subsets of the same data and would double-count rows.
NSLKDD_CANONICAL_FILES = {"KDDTrain+.txt", "KDDTest+.txt"}
# UNSW-NB15_1..4.csv are raw headerless flow batches with no aligned ground-truth
# label file in this mirror -- mixing them with the labeled pre-split files below
# would blend two different schemas into meaningless row/column/class totals.
UNSW_CANONICAL_FILES = {"UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"}

def find_tabular_files(root):
    matches = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith((".csv", ".txt")):
                matches.append(os.path.join(dirpath, fn))
    # some archives (e.g. this NSL-KDD Kaggle zip) nest a duplicate copy of every
    # file inside a subfolder -- keep only the shallowest path per basename so
    # counts aren't silently doubled
    by_basename = {}
    for p in matches:
        fn = os.path.basename(p)
        if fn not in by_basename or p.count(os.sep) < by_basename[fn].count(os.sep):
            by_basename[fn] = p
    return list(by_basename.values())

for name, info in dataset_status.items():
    if info["status"] != "ready":
        continue

    files = find_tabular_files(info["path"])
    if name == "nsl-kdd":
        files = [p for p in files if os.path.basename(p) in NSLKDD_CANONICAL_FILES]
    elif name == "unsw-nb15":
        files = [p for p in files if os.path.basename(p) in UNSW_CANONICAL_FILES]
    if not files:
        info["issues"] = ["No CSV/TXT files found after extraction -- check archive contents."]
        print(f"[{name}] No tabular files found under {info['path']}")
        continue

    total_rows, n_cols, label_counts, issues = 0, None, {}, []

    for path in files:
        fname = os.path.basename(path)
        try:
            seen_hashes, duplicates_found = set(), 0
            dup_check_active, rows_for_dup_check = True, 0
            read_kwargs = {"chunksize": 100_000, "low_memory": False, "on_bad_lines": "skip"}
            if name == "nsl-kdd":
                read_kwargs.update(header=None, names=NSLKDD_COLUMNS)
            for chunk in pd.read_csv(path, **read_kwargs):
                total_rows += len(chunk)
                if n_cols is None:
                    n_cols = chunk.shape[1]
                label_col = next((c for c in LABEL_COLUMN_CANDIDATES if c in chunk.columns), None)
                if label_col:
                    for k, v in chunk[label_col].value_counts().items():
                        label_counts[str(k)] = label_counts.get(str(k), 0) + int(v)
                if chunk.isnull().any().any():
                    issues.append(f"missing values in {fname}")
                if dup_check_active:
                    rows_for_dup_check += len(chunk)
                    if rows_for_dup_check > DUP_CHECK_ROW_LIMIT:
                        dup_check_active = False
                        issues.append(f"duplicate-row check skipped for {fname} (exceeds {DUP_CHECK_ROW_LIMIT:,} rows)")
                    else:
                        for row_hash in pd.util.hash_pandas_object(chunk, index=False):
                            if row_hash in seen_hashes:
                                duplicates_found += 1
                            else:
                                seen_hashes.add(row_hash)
            if dup_check_active and duplicates_found:
                issues.append(f"{duplicates_found} duplicate rows in {fname}")
        except Exception as e:
            issues.append(f"failed to parse {fname}: {e}")

    info["row_count"] = total_rows
    info["column_count"] = n_cols
    info["class_distribution"] = label_counts
    info["issues"] = sorted(set(issues))

    print(f"[{name}] rows={total_rows:,} cols={n_cols} classes={label_counts or 'not detected'}")
    if info["issues"]:
        print(f"  issues: {info['issues']}")

## 9. Write dataset manifest

In [ ]:
import datetime

manifest = {
    "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
    "datasets": dataset_status,
}

manifest_path = "data/dataset_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {manifest_path}")

## 10. Save to Drive + push to GitHub

In [ ]:
for name, info in dataset_status.items():
    if info["status"] != "ready":
        continue
    drive_target = os.path.join(DRIVE_DATA_RAW, name)
    if not (os.path.isdir(drive_target) and any(os.scandir(drive_target))):
        print(f"Backing up {name} to Drive...")
        shutil.copytree(info["path"], drive_target, dirs_exist_ok=True)

shutil.copy2(manifest_path, os.path.join(DRIVE_ROOT, "data", "dataset_manifest.json"))
print("Drive backup complete.")

In [ ]:
subprocess.run(
    ["git", "-C", REPO_DIR, "add",
     "notebooks/00_Setup_and_Data.ipynb", ".gitignore", "data/dataset_manifest.json"],
    check=True,
)

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "commit", "-m", "Notebook 00: setup + dataset acquisition"],
    capture_output=True, text=True,
)
print(commit.stdout, commit.stderr)

if commit.returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
    print("Pushed to GitHub.")
else:
    print("Nothing new to commit (or commit failed) -- see output above.")

## 11. Final summary

Paste this cell's output back for review before Notebook 01 starts.

In [ ]:
print("=" * 70)
print("CLEIDS-Edge -- Notebook 00 Summary")
print("=" * 70)

ready = [n for n, i in dataset_status.items() if i["status"] == "ready"]
manual = [n for n, i in dataset_status.items() if i["status"] == "manual_required"]
failed = [n for n, i in dataset_status.items() if i["status"] == "failed"]

print(f"\nReady ({len(ready)}):")
for n in ready:
    info = dataset_status[n]
    print(f"  - {n}: {info.get('row_count', '?')} rows, {info.get('column_count', '?')} cols, "
          f"source={info['source_type']} ({info['source']})")

print(f"\nNeeds manual action ({len(manual)}):")
for n in manual:
    info = dataset_status[n]
    print(f"  - {n}: {info['note']}")
    print(f"    -> {info['instructions']}")

print(f"\nFailed ({len(failed)}):")
for n in failed:
    info = dataset_status[n]
    print(f"  - {n}: {info['error']}")

print("\nManifest written to data/dataset_manifest.json and pushed to GitHub.")
print("Next: once manual datasets are resolved and this cell shows all datasets 'ready', "
      "re-run this notebook once to pick them up, paste this summary back for review, then "
      "proceed to Notebook 01 (Preprocessing_and_Feature_Engineering).")